In [ ]:
import cv2
import json
import gradio as gr
from ultralytics import YOLO
from PIL import Image
#import time

model = YOLO("weights/best.pt") # YOLO11s 50 epochs 
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # opencv use default webcam with DirectShow (faster and reliable)

min_stable_frames = 15 # minimum frames needed for the item to be consistently detected (no flicker detection)
min_clear_table = 20 # minimum frames the table must be clear before scanning items

previous_frame = {} # dict to store items and their counts from last frame
stable_frames = 0
clear_table = 0
confirmed_batch = None # save items detected after the min_stable_frames is reached (save for scanning)
status_text = "Load a receipt to start" 
receipt_text = ""
live_frame = None 
ordered = {} # dict storing receipt info

# Take input receipt.json file, loop through it to make ordered
def load_receipt(receipt_file):
    # set variables as global, if not python makes a new local variable everytime you try edit them
    global ordered, receipt_text, status_text, confirmed_batch
    
    receipt = json.load(open(receipt_file.name)) # load inputted receipt file

    # clear any stale data
    confirmed_batch = None
    ordered = {}
    
    # receipt looks like [item 1, item 1, item 2, item 3, item 4] --> duplicate is on purpose
    for item in receipt: # loop through list
        if item in ordered: # if item already exists in ordered
            ordered[item] += 1 # add one to its count
        else: # if its a new item
            ordered[item] = 1 # initialise new key with value of 1
            
    receipt_text = build_receipt() # show the loaded receipt
    status_text = "Ready" # set status to ready  

# Build the receipt display string (markdown formatted)
def build_receipt():
    lines = [] # build list of strings then join at the end
    for name, count in ordered.items():
        if count == 0:
            lines.append("~~" + name + "~~") # strikethrough for scanned items
        else:
            lines.append(name + " - " + str(count) + " left") # show remaining count
    return "  \n".join(lines) # two spaces before \n is markdown line break

# function for Gradio to poll, returns the global values (so nothing needs to be passed in)
def get_status():
    return live_frame, status_text, receipt_text

# When "Done" is clicked, list missing items
def finish():
    global status_text
    if not ordered: # no receipt loaded, nothing to check
        status_text = "Load a receipt first"
        return # return lives INSIDE the if, so it only exits early on an empty receipt
    missing = []
    for name, count in ordered.items():
        if count != 0: # count > 0 means missing item
            missing.append(str(count) + " x " + name)
    if missing:
        status_text = "Not Delivered: " + ", ".join(missing)
    else:
        status_text = "Delivery verified!"

# For photo uploads, same verify against receipt logic
def verify(images, receipt_file):
    if not images or not receipt_file: 
        return [], "Upload photos and a receipt first"
    receipt = json.load(open(receipt_file.name)) # load receipt
    # turn receipt.json contents into a dictionary
    wanted = {} # instead of ordered so both can be used at once in demo 
    for item in receipt:
        if item in wanted:
            wanted[item] += 1
        else:
            wanted[item] = 1

    annotated = [] # list of images with bounding boxes draw on
    found = {} # dict of item: count
    for f in images: # loop through the inputted images
        res = model(Image.open(f.name), conf=0.8)[0] # pass to model, min confidence=0.8, returns YOLO results object
        annotated.append(res.plot()[:, :, ::-1]) # [h, w, BGR] (from .plot()) --> [h, w, RGB] (for Gradio display)
        for c in res.boxes.cls: # cls is a tensor of class indices like [0, 2, 0....], loop through it
            name = model.names[int(c)] # int(tensor element) = plain integer, model.names() maps to actual class name
            if name in found:
                found[name] += 1
            else:
                found[name] = 1

    lines = []
    for item, qty in wanted.items(): # subtract found items from receipt
        if item in found:
            diff = qty - found[item]
        else:
            diff = qty
        if diff > 0:
            lines.append("- " + str(diff) + " x " + item) # add to lines only if items are missing
    if lines: # if lines has any content, that means items were missing
        return annotated, "**You are missing:**  \n" + "  \n".join(lines) # return annotated images and missing items
        
    return annotated, "**Delivery verified!**" # return annotated images and confirmation message

# web UI code
with gr.Blocks(css="#receipt, #receipt * {font-size: 26px !important}") as app:
    gr.Markdown("# FairPrice Delivery Verification")

    gr.Markdown("### Live mode")
    with gr.Row():
        cam = gr.Image(label="Camera")
        with gr.Column():
            live_receipt_in = gr.File(label="e-Receipt (.json)")
            receipt_box = gr.Markdown("", elem_id="receipt") # elem_id so the css above can target this box

    status_box = gr.Label(value="Load a receipt to start")
    done_btn = gr.Button("Done - check my order")

    gr.Markdown("### Photo mode")
    with gr.Row():
        img_in = gr.File(file_count="multiple", file_types=["image"], label="Photos of your items")
        receipt_in = gr.File(label="e-Receipt (.json)")
    verify_btn = gr.Button("Verify")
    with gr.Row():
        img_out = gr.Gallery(label="Detections")
        text_out = gr.Markdown()

    live_receipt_in.upload(load_receipt, live_receipt_in, None) # upload listener, (fn, inputs, outputs)
    done_btn.click(finish, None, None) # click listener, same format (fn, inputs, outputs)
    verify_btn.click(verify, [img_in, receipt_in], [img_out, text_out])
    # on load, (fn, inputs, outputs, param1, param2), run this every 0.2s and hide the loading spinner
    app.load(get_status, None, [cam, status_box, receipt_box], every=0.2, show_progress="hidden")

app.launch(prevent_thread_lock=True) # launch app but don't stop the program code from running

"""
try, finally ensures that the cleanup code always runs, putting it in a try means that even if the code crashes or
keyboard interrupt happens, the finally block still triggers allowing the cleanup code to run
"""

try:
    while True: # loop forever
        ret, frame = cap.read()
        if not ret:
            break

        # using webcam but I need to tilt it down, so to crop out the keyboard and me I only take the middle 70%
        h = frame.shape[0] # get height of video frame
        frame = frame[int(h * 0.15):int(h * 0.85)] # crop to 70%

        # pass each frame to the model with a min confidence of 0.8, verbose=False stops YOLO from printing speed and other info so my terminal doesn't get extremely messy
        r = model(frame, conf=0.8, verbose=False)[0] 
        live_frame = r.plot()[:, :, ::-1]

        ## opencv camera was used for testing because Gradio one is SO LAGGY
        # cv2.imshow("scan - q to quit", r.plot())
        # if cv2.waitKey(1) & 0xFF == ord("q"):
        #     break

        items = {}
        for c in r.boxes.cls: # loop through tensor of class ids
            name = model.names[int(c)] # get class name
            if name in items:
                items[name] += 1
            else:
                items[name] = 1

        """
        Main tracking logic: In order to avoid phantom detections and also allow for multiple items to be detected,
        I keep track of how many frames the detections are stable for. If, for min_stable_frames the detections are stable,
        I can confidently say the items being detected are valid. 
        In order to know the detections are stable, i need to make sure that the detections are consistent with the previous frame
        Once the detections are identified as stable, i save the current detections in a confirmed_batch
        confirmed_batch only changes once another set of stable detections are identified, this allows you to change the detected items just in case
        Before committing the scan, i make sure that all items are removed from the table to stop accidental rescanning of the same objects from slow removal
        To check that, clear_table needs to reach min_clear_table
        Once that happens, I can remove items from the receipt based on confirmed_batch, then display that using build_receipt()
        """
        
        if items:
            clear_table = 0 # item detected = table not clear
            if items == previous_frame: # check if the detections are the same as last frame
                stable_frames += 1 # count as stable frame
            else:
                stable_frames = 1 # if not reset to 1
                previous_frame = items # set previous_frame to current frames items
            # == instead of >= so that it doesn't trigger multiple times on the same valid scan, it HAS to reset before rescanning
            if stable_frames == min_stable_frames:
                confirmed_batch = dict(items)
                # print("CONFIRMED:", confirmed_batch) ## testing code
                status_text = "Confirmed: " + str(confirmed_batch)
        else: 
            # no items = reset vars and increment clear_table
            stable_frames = 0
            previous_frame = {}
            clear_table += 1

            # if table is clear for long enough and there is a confirmed_batch ready to be processed
            if clear_table == min_clear_table and confirmed_batch is not None and ordered: # and ordered stops a scan from committing before a receipt is even loaded
                # print(time.strftime("%H:%M:%S"), "SCANNED:", confirmed_batch) ## testing code with timestamp for tracking
                
                status_text = "Scanned: " + str(confirmed_batch) # display scan success
                for key, value in confirmed_batch.items(): # loop through detected items
                    if key in ordered: # if the item is in the receipt
                        new_val = ordered[key] - value # subtract the count from the receipt
                        if new_val < 0: # in case of extra items, mainly for testing but protects from negative counts
                            ordered[key] = 0
                        else:
                            ordered[key] = new_val
                receipt_text = build_receipt() # update receipt
                confirmed_batch = None # reset batch

                # if all items are scanned, they are all delivered
                done = True 
                for value in ordered.values():
                    if value != 0:
                        done = False
                if done:
                    status_text = "Delivery verified!"
finally:
    cap.release()
    # cv2.destroyAllWindows()